In [2]:
import dotenv
from langchain_openai import ChatOpenAI
from datetime import datetime
import os
import json
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

dotenv.load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["OPENAI_BASE_URL"] = os.getenv("OPENAI_BASE_URL")
llm = ChatOpenAI(model="gpt-4o-mini")

class LLMClient:
    def __init__(self, api_key=None, base_url=None, model=None, temperature=None):
        dotenv.load_dotenv()    #加载环境变量

        self.api_key = api_key or os.getenv("OPENAI_API_KEY")   #优先使用传进来的参数作为key和url
        self.base_url = base_url or os.getenv("OPENAI_BASE_URL")

        if not self.api_key:
            raise ValueError("API Key 不能为空！请检查：1) .env文件 2) 环境变量")
        if not self.base_url:
            raise ValueError("Base URL 不能为空！请检查 .env 文件")

        self.model = model or "gpt-4o-mini"
        self.temperature = temperature or 0.5

        try:
            self.llm = ChatOpenAI(
                model=self.model,
                api_key=self.api_key,
                base_url=self.base_url,
                temperature = self.temperature
            )
        except Exception as e:
            raise RuntimeError(f"初始化 LLM 失败：{e}")

        self.history=[] #存储对话历史记录

    def historyChat(self, prompt):
        # 将字符串 prompt 转为 HumanMessage
        self.history.append(HumanMessage(content=prompt))

        max_history = 20
        while len(self.history) > max_history:
            self.history.pop(0)
            self.history.pop(0)

        # 直接传 history 列表给 llm.invoke()
        response = self.llm.invoke(self.history)
        self.history.append(AIMessage(content=response.content))
        return response.content

    def clear_history(self):
        self.history = []
        return

    def save_history(self, filepath=None):
        if filepath is None:
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            filepath = f"history_{timestamp}.json"

        data = {
            "save_time": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "model": self.model,
            "message_count": len(self.history),
            "messages": self.history
        }

        # 写入文件
        with open(filepath, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=2)

        print(f"对话已保存到：{filepath}")
        return

    def load_history(self, filepath):

        if not os.path.exists(filepath):
            print(f"文件不存在：{filepath}")
            return False

        # 读取JSON文件
        with open(filepath, "r", encoding="utf-8") as f:
            data = json.load(f)

        # 恢复历史记录
        self.history = data.get("messages", [])
        return

In [4]:
client = LLMClient(temperature=0)
print(client.historyChat("介绍下你的模型"))

我是基于OpenAI的GPT-3模型，属于生成式预训练变换器（Generative Pre-trained Transformer）系列。我的设计目的是理解和生成自然语言文本，能够进行对话、回答问题、提供信息、撰写文章等。

我的模型通过大量的文本数据进行训练，学习语言的结构、语法、语义和上下文关系。这使得我能够在多种主题上进行交流，并提供相关的知识和建议。

如果你有任何具体的问题或需要了解的内容，欢迎随时问我！


# 实现一个NLU
## 定义任务和输入

In [8]:
from langchain_core.prompts import ChatPromptTemplate

# 任务描述
instruction = """
你的任务是识别用户对手机流量套餐产品的选择条件。
每种流量套餐产品包含三个属性:名称，月费价格，月流量。
根据用户输入，识别用户在上述三种属性上的倾向。
"""

# 用户输入
user_input = "办个100G的套餐。"  # 注意：避免用 input 作为变量名（内置函数）


prompt_template = ChatPromptTemplate([
    ("system", instruction),
    ("human", "{user_input}")
])

prompt = prompt_template.format(user_input=user_input)

print(client.historyChat(prompt))

NameError: name 'client' is not defined

# 限定输出格式：JSON 用提示词限定

In [6]:
# 用提示词限定
output = """
以json格式输出内容
"""
prompt_template = ChatPromptTemplate([
    ("system", instruction+output),
    ("human", "{user_input}")
])
prompt = prompt_template.format(user_input=user_input)
print(client.historyChat(prompt))


```json
{
  "选择条件": {
    "名称": null,
    "月费价格": null,
    "月流量": "100G"
  }
}
```


In [5]:
zero_shot = LLMClient()
few_shot = LLMClient()

prompt_zero_shot = """
分析以下文本的情感倾向（正面/负面/中性），并给出置信度0-1：

文本："这家餐厅环境优雅，服务员态度也很好，但是等位等了两个小时，菜品上来已经凉了，价格还贵得离谱。不过甜品确实惊艳，让人又爱又恨。"

请输出：
情感：
置信度：
分析：
"""

prompt_few_shot = """
分析文本情感倾向，参考以下示例：

示例1：
文本：这家餐厅服务超好，菜品也很棒！
情感：正面
置信度：0.95

示例2：
文本：等了一个小时，结果菜还是凉的，气死我了
情感：负面
置信度：0.9

示例3：
文本：价格一般，味道还行，无功无过吧
情感：中性
置信度：0.7

现在分析：
文本："这家餐厅环境优雅，服务员态度也很好，但是等位等了两个小时，菜品上来已经凉了，价格还贵得离谱。不过甜品确实惊艳，让人又爱又恨。"
情感：
置信度：
分析：
"""

response_few_shot = few_shot.historyChat(prompt_few_shot)
response_zero_shot = zero_shot.historyChat(prompt_zero_shot)

print(response_zero_shot)
print("-----------")
print(response_few_shot)

情感：中性  
置信度：0.7  

分析：该文本包含了对餐厅的多个方面的评价，既有正面的描述（如“环境优雅”、“服务员态度很好”、“甜品确实惊艳”），也有负面的反馈（如“等位等了两个小时”、“菜品上来已经凉了”、“价格还贵得离谱”）。由于正面和负面情感的内容相互交织，整体情感倾向趋向中性。置信度为0.7，反映了文本情感的复杂性和多样性。
-----------
情感：中性  
置信度：0.75  

分析：文本中提到的“环境优雅”和“服务员态度很好”传达了正面的情感，但同时也提到“等位等了两个小时”和“菜品上来已经凉了，价格还贵得离谱”，这些内容则带有负面的情感。最后，提到“甜品确实惊艳”又为整体情感增添了一些积极的色彩。因此，整体情感较为复杂，表现出中性倾向，置信度为0.75。


## 思维链CoT 让模型一步步思考
思维链提示
思维链提示，就是把一个多步骤推理问题，分解成很多个中间步骤，分配给更多的计算量，生成更多的 token，再把这些答案拼接在一起进行求解。

论文里面作者提到了很多 CoT 的优势，其中包括它把一个多步推理问题分解出多个中间步骤，并且让 LLM 更加可解释。它能解决的问题很多，除了上述的数学应用题，还有常识推理、以及 symbolic manipulation （符号操作）这类任务（就是一些手造的考验大模型的问题，比如最典型的 Last Letter Concatenation（最后一个字母串联） 和 coin flip（抛硬币）），下面补充几个例子方便理解：

In [6]:
lmm_withoutCoT = LLMClient()
lmm_withCoT = LLMClient()

prompt_without_cot = """
题目：甲、乙、丙三人中，一人是医生，一人是教师，一人是工程师。已知：

甲比医生年龄大
乙和教师不同岁
教师比丙年龄小
问：三人分别是什么职业？
"""
prompt_with_cot = """
题目：甲、乙、丙三人中，一人是医生，一人是教师，一人是工程师。已知：

甲比医生年龄大
乙和教师不同岁
教师比丙年龄小
问：三人分别是什么职业？
请一步一步思考：
"""

print(lmm_withoutCoT.historyChat(prompt_without_cot))
print(lmm_withCoT.historyChat(prompt_with_cot))

根据题目中的信息，我们可以逐步推理出三人的职业。

1. **甲比医生年龄大**：这说明甲不是医生。
2. **乙和教师不同岁**：这说明乙不是教师。
3. **教师比丙年龄小**：这说明教师不是丙。

根据以上信息，我们可以进行如下推理：

- 由于甲不是医生，乙也不是教师，所以丙必须是医生。
- 因为丙是医生，教师比丙年龄小，那么教师只能是甲。
- 这样，乙就只能是工程师。

因此，三人的职业分别是：
- 甲：教师
- 乙：工程师
- 丙：医生
我们来逐步分析这个问题。

1. **确定职业**：甲、乙、丙三个人分别是医生、教师和工程师。

2. **分析条件**：
   - 条件1：甲比医生年龄大。
   - 条件2：乙和教师不同岁。
   - 条件3：教师比丙年龄小。

3. **从条件1开始**：
   - 如果甲比医生年龄大，那么甲不可能是医生。因为如果甲是医生，那么他和医生的年龄就是相同的，不符合条件1。因此，甲可以是教师或工程师。

4. **考虑条件3**：
   - 教师比丙年龄小。假设教师是乙，那么丙的年龄必须大于乙的年龄，这与条件2（乙和教师不同岁）相矛盾。因此，教师不能是乙。

   由此，我们可以推断出：
   - 教师是甲。
   - 由于甲是教师，丙的年龄必须大于甲的年龄。

5. **总结当前信息**：
   - 甲是教师。
   - 丙的年龄大于甲（教师）。
   - 由于甲是教师，医生只能是乙。

6. **确定丙的职业**：
   - 由此，丙只能是工程师。

7. **最终结果**：
   - 甲：教师
   - 乙：医生
   - 丙：工程师

因此，三人的职业分别是：
- 甲：教师
- 乙：医生
- 丙：工程师


## 结合上述规则优化提示词，实现一个NLU

In [8]:
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate
cli = LLMClient()


instruction = """
你的任务是识别用户对手机流量套餐产品的选择条件。
每种流量套餐产品包含三个属性:名称【name】，月费价格【price】，月流量【data】。
根据用户输入，识别用户在上述三种属性上的倾向。
"""

output_format = """
以JSON格式输出。
1. name字段的取值为string类型，取值必须为以下之一：经济套餐、畅游套餐、无限套餐、校园套餐 或 null；

2. price字段的取值为一个结构体 或 null，包含两个字段：
(1) operator, string类型，取值范围：'<='（小于等于）, '>=' (大于等于), '=='（等于）
(2) value, int类型

3. data字段的取值为取值为一个结构体 或 null，包含两个字段：
(1) operator, string类型，取值范围：'<='（小于等于）, '>=' (大于等于), '=='（等于）
(2) value, int类型或string类型，string类型只能是'无上限'

4. 用户的意图可以包含按price或data排序，以sort字段标识，取值为一个结构体：
(1) 结构体中以"ordering"="descend"表示按降序排序，以"value"字段存储待排序的字段
(2) 结构体中以"ordering"="ascend"表示按升序排序，以"value"字段存储待排序的字段

输出中只包含用户提及的字段，不要猜测任何用户未直接提及的字段，不输出值为null的字段。
"""


# 1. 定义示例列表（必须是字典列表）
examples_list = [
    {"input": "便宜的套餐", "output": '{"sort":{"ordering":"ascend","value":"price"}}'},
    {"input": "有没有不限流量的", "output": '{"data":{"operator":"==","value":"无上限"}}'},
    {"input": "流量大的", "output": '{"sort":{"ordering":"descend","value":"data"}}'},
    {"input": "100G以上流量的套餐最便宜的是哪个", "output": '{"sort":{"ordering":"ascend","value":"price"},"data":{"operator":">=","value":100}}'},
    {"input": "月费不超过200的", "output": '{"price":{"operator":"<=","value":200}}'},
    {"input": "就要月费180那个套餐", "output": '{"price":{"operator":"==","value":180}}'},
    {"input": "经济套餐", "output": '{"name":"经济套餐"}'},
    {"input": "土豪套餐", "output": '{"name":"无限套餐"}'},
]

# 2. 定义单个示例的格式化模板
example_prompt = ChatPromptTemplate.from_messages([
    ("human", "{input}"),
    ("ai", "{output}"),
])

# 3. 创建 few-shot 模板
few_shot_template = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples_list,  # 传入列表，不是字符串
)

# 4. 组合最终模板
final_prompt = ChatPromptTemplate.from_messages([
    ("system", instruction + output_format),
    few_shot_template,
    ("human", "{user_input}"),
])

# 5. 使用
user_input = "办个100G的套餐。"
prompt = final_prompt.format(user_input=user_input)
print(cli.historyChat(prompt))

{"data":{"operator":">=","value":100}}
